# Question 5 — Constellation detection (second attempt)

Each scene is a 3000×3000 star field and a folder of 32×32 patches. A patch is centred on a star in this field, or it comes from somewhere else. The score per scene is

`0.25 · presence + 0.20 · localisation + 0.25 · figure recovery + 0.30 · constellation name`.

**Training score (three labelled scenes): 0.405 → 0.796.** All three constellations are now named correctly, and figure recovery is 0.78 / 1.00 / 1.00.

Code lives in three modules so the notebook and the submitted script run the same thing:

- `q5_search.py` — stage 1, the correlation sweep;
- `q5_identify.py` — stage 2, registering pattern drawings to the sky;
- `q5_v2.py` — stage 3, assigning figure stars, present/absent, and the CSV.


## What the first attempt got wrong

Three properties of the data broke the first version.

1. **Local appearance is ambiguous on purpose.** Bright stars are saturated round disks, so a figure-star patch correlates at 0.95+ with many bright stars. A few figure-star neighbourhoods are also pasted elsewhere as exact pixel copies. Taking the single highest correlation put many figure stars hundreds of pixels away.
2. **Off-figure stars can form a decoy constellation.** In the Taurus scene, 11 of the 12 off-figure stars line up with Orion's drawing. Fitting a drawing to "the recovered points" named Orion.
3. **The fit was run on wrong points.** Once localisation failed, the name was chosen by whichever drawing touched the errors.


## Stage 1 — Keep every near-tied location (lecture 1: template matching)

Each patch is warped through 36 rotations (10° apart) and 3 scales, and normalised cross-correlation is computed against the whole sky with `cv2.matchTemplate`. Instead of the single best peak, up to 16 distinct locations are kept per patch. A parabola on the correlation peak gives sub-pixel centres.

On the training scenes the true location of a figure star is in this list for 8 of 10 Pisces figure stars, usually within 0.01 of the best correlation. It is ranked first much less often. That is what stage 2 is for.


## Stage 2 — Register the drawings to the candidates (lectures 2–3: matching, RANSAC, transforms)

Pattern PNGs are white nodes on green edges; the nodes are extracted as connected components.

The candidate cloud is every location within 0.03 of a patch's best correlation, restricted to **bright sky** (top 20% of local star brightness). Figure stars are always bright, and this removes most of the noise.

A geometric hash then finds, for each drawing, the similarity transform (with optional reflection) that puts the most nodes on candidates. Every ordered pair of candidates from two different patches defines a coordinate frame; pairs of pattern nodes vote for frames in which the rest of the drawing lands on other candidates. The best frames are refit by least squares on their inliers. **Each patch can supply at most one inlier**, so several guesses from the same patch cannot all count.


## Stage 3 — Verify by claiming nodes, then assign

The five drawings with the most inliers are each verified directly. For every predicted node, every candidate patch is swept through all rotations in a small window around the node; a one-to-one assignment between nodes and patches (Hungarian algorithm) keeps claims with correlation ≥ 0.70.

The drawing with the highest **average claim strength per node** wins. Dividing by node count matters: large drawings like Hydra or Eridanus otherwise win by offering more bright targets. On the true drawing nearly every node is a bright star that a patch matches closely.

Claimed patches are reported at the claimed location with `m = 1`. Every other patch is present (`m = 0`, at its best peak) if its best correlation is ≥ 0.90, or ≥ 0.55 with a 0.04 margin over the next distinct location; otherwise `-1`.

Figure recovery is scored against *all* present points, not per patch, so a node claimed by any patch that correlates there still earns credit.


In [ ]:
from pathlib import Path
import pandas as pd
import q5_v2, q5_search
from q5_identify import load_patterns

patterns = load_patterns(q5_v2.ROOT / "patterns")
print(len(patterns), "patterns;", len(q5_search.ANGLES), "rotations x", len(q5_search.SCALES), "scales")
print(q5_v2.PARAMS)


## Training score

Stage 1 results are cached under `cache_v2/` after the first run (about 5 minutes per scene on 7 cores). Stage 3's local sweep takes about 2 minutes per scene.


In [ ]:
q5_v2.evaluate_training(patterns)


## Validation submission


In [ ]:
out = Path("submission_v2.csv")
q5_v2.write_submission(patterns, out)
print(pd.read_csv(out)[["Id", "n_patches", "constellation"]].to_string(index=False))


## Limits

- Parameters were checked on only three labelled scenes. The selection rule and presence thresholds sit in the middle of broad plateaus rather than at a single best value, but the hidden scenes may still behave differently.
- Off-figure stars are still located by their single best peak, so field-star localisation stays near 0.3–0.4.
- A decoy constellation made of genuinely bright field stars can still win if the true figure's nodes are poorly localised.
